# AIaaS — Embeddings / RAG Throughput Benchmark

Portable throughput benchmark for the **embeddings** workload (the encode step
behind RAG / semantic search). Sweeps batch size and reports how many
sentences/s and tokens/s the GPU sustains, plus latency and VRAM.

Reports per batch size:
- **sentences/s** and **tokens/s** (encode throughput)
- **ms per batch** (latency)
- **peak VRAM**

> Tier: *portable proxy* — a clean `sentence-transformers` timing run, not an
> accuracy-gated MLPerf number. Runs on any GPU (tiny model). For the credible
> tier, optimum-benchmark / MLPerf BERT is the next rung.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "sentence-transformers", "pandas"], check=False)
import sentence_transformers, transformers
print("sentence-transformers", sentence_transformers.__version__,
      "| transformers", transformers.__version__)


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."
_cc = torch.cuda.get_device_capability(0)

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration

In [ ]:
MODEL = "BAAI/bge-small-en-v1.5"   # ungated, ~130 MB; runs anywhere
BATCH_SIZES = [1, 8, 32, 128]
SEQ_WORDS = 64                    # approx tokens/sentence via repeated words
NUM_SENTENCES = 2000             # corpus size encoded per batch-size point
WARMUP_BATCHES = 3
DTYPE = "float16" if _cc[0] >= 7 else "float32"

OUT_DIR = "embeddings_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL={MODEL}, batch_sizes={BATCH_SIZES}, dtype={DTYPE}")


## 4. Build a synthetic corpus
Fixed-length sentences so tokens/s is comparable across runs.

In [ ]:
import random
random.seed(0)
_words = ("retrieval augmented generation embeddings vector search semantic "
          "index nearest neighbour cosine similarity transformer encoder").split()
def make_sentence(n_words):
    return " ".join(random.choice(_words) for _ in range(n_words))
CORPUS = [make_sentence(SEQ_WORDS) for _ in range(NUM_SENTENCES)]
print("corpus:", len(CORPUS), "sentences; sample:\n", CORPUS[0][:120])


## 5. Load model + run the batch-size sweep

In [ ]:
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL, device="cuda")
if DTYPE == "float16":
    model = model.half()

# token count for tokens/s (encode the corpus once through the tokenizer)
tok = model.tokenizer
avg_tokens = sum(len(tok.encode(s)) for s in CORPUS[:200]) / 200.0

def run_bs(bs):
    torch.cuda.reset_peak_memory_stats()
    # warmup
    model.encode(CORPUS[:bs * WARMUP_BATCHES], batch_size=bs,
                 show_progress_bar=False, convert_to_numpy=True)
    torch.cuda.synchronize()
    t0 = time.time()
    model.encode(CORPUS, batch_size=bs, show_progress_bar=False, convert_to_numpy=True)
    torch.cuda.synchronize()
    dt = time.time() - t0
    n_batches = (len(CORPUS) + bs - 1) // bs
    return {
        "batch_size": bs,
        "sentences_per_s": round(len(CORPUS) / dt, 1),
        "tokens_per_s": round(len(CORPUS) * avg_tokens / dt, 1),
        "ms_per_batch": round(1000.0 * dt / n_batches, 2),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

RESULTS = []
for bs in BATCH_SIZES:
    r = run_bs(bs)
    print(r)
    RESULTS.append(r)


## 6. Results — normalize, save, summarize

In [ ]:
import pandas as pd
NORM = {
    "schema": "embeddings-bench/1.0",
    "env": ENV, "model": MODEL, "dtype": DTYPE,
    "seq_words": SEQ_WORDS, "avg_tokens_per_sentence": round(avg_tokens, 1),
    "num_sentences": NUM_SENTENCES, "results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"embeddings_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== EMBEDDINGS SUMMARY ====\n{ENV['gpu_name']} | {MODEL} | {DTYPE}")
df = pd.DataFrame(RESULTS)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 7. Reading the result
Throughput rises with batch size until the GPU saturates; the **knee** is the
efficient serving batch size. `sentences/s` sizes a RAG ingest/query pipeline.
For credible cross-framework numbers, run the same model through
`optimum_crossframework_benchmark.ipynb`.